# Transfer Learning με BERT: Fine-Tuning μέσω του CLS Token

Σε αυτό το notebook κάνουμε fine-tune ένα pretrained μοντέλο BERT για ένα πρόβλημα δυαδικής ταξινόμησης (binary classification) κειμένου. Ο στόχος μας είναι να γίνουν ορατά τα βήματα του transfer learning: αντί να χρησιμοποιήσουμε wrappers του HuggingFace που κρύβουν τις λεπτομέρειες, φορτώνουμε απευθείας τον κορμό του BERT, εξετάζουμε την αρχιτεκτονική του, και συνδέουμε τον ταξινομητή (classifier) μόνοι μας, ακριβώς όπως κάναμε και με την ResNet [σε αυτή την διάλεξη](https://youtu.be/0NfdJbMd3eM).


Το πρόβλημα ταξινόμησης είναι το **CoLA benchmark** (Corpus of Linguistic Acceptability — Σώμα Γλωσσολογικής Αποδεκτότητας): δεδομένης μιας αγγλικής πρότασης, να προβλεφθεί αν είναι γραμματικά και γλωσσικά αποδεκτή. Η βασική ιδέα είναι ότι το BERT, έχοντας εκπαιδευτεί εκ των προτέρων σε μεγάλο σώμα κειμένου, έχει ήδη αναπτύξει πλούσιες γλωσσολογικές αναπαραστάσεις -- έχει αναπτύξει μια τεχνητή "κατανόηση" της γλώσσας. Μπορούμε να επαναχρησιμοποιήσουμε αυτές τις αναπαραστάσεις για ένα νέο πρόβλημα, προσθέτοντας έναν απλό ταξινομητή που αποτελείται από ένα τελευταίο γραμμικό επίπεδο ("classification head") και κάνοντας fine-tuning.

```
Κωνσταντίνος Καραμανής: constantine@utexas.edu
http://users.ece.utexas.edu/~cmcaram/
The University of Texas at Austin
Archimedes/Athena RC
```


In [ ]:
!pip install -q datasets

In [ ]:
import torch
import transformers
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModelForMaskedLM
from datasets import load_dataset
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Sequence, Dict

## Το Dataset: CoLA

Το CoLA (Corpus of Linguistic Acceptability — Σώμα Γλωσσολογικής Αποδεκτότητας) είναι ένα πρόβλημα δυαδικής ταξινόμησης από το σύνολο αξιολόγησης GLUE. Κάθε παράδειγμα είναι μια αγγλική πρόταση με ετικέτα 1 (γραμματικά αποδεκτή) ή 0 (μη αποδεκτή), αντλημένη από συντακτικά εγχειρίδια και επιστημονικά άρθρα. Εξετάζει κατά πόσο ένα μοντέλο έχει εσωτερικεύσει γνώση αγγλικής γραμματικής πέρα από απλή αναγνώριση επιφανειακών μοτίβων.

In [ ]:
dataset = load_dataset("nyu-mll/glue", "cola")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1043 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1063 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1063
    })
})


### Τι Περιέχει;

Όπως πάντα, κοιτάμε τα δεδομένα που κατεβάσαμε.

In [ ]:
print(dataset["train"][0])
print(dataset["train"][10])
print(dataset["train"][41])
print(dataset["train"][42])

{'sentence': "Our friends won't buy this analysis, let alone the next one we propose.", 'label': 1, 'idx': 0}
{'sentence': 'The critics laughed the play off the stage.', 'label': 1, 'idx': 10}
{'sentence': 'They caused him to become president by making him.', 'label': 0, 'idx': 41}
{'sentence': 'They made him to exhaustion.', 'label': 0, 'idx': 42}


## Tokenization

Το BERT χρησιμοποιεί WordPiece tokenization, η οποία κατατέμνει το κείμενο σε υπολεξιλογικές μονάδες από ένα σταθερό λεξιλόγιο 30.522 tokens. Σημειώνουμε τρεις σημαντικές λεπτομέρειες:


- **[CLS] token (id 101)**: τοποθετείται στην αρχή κάθε ακολουθίας. Μετά τη διέλευση από τον πλήρη transformer encoder, η κρυφή κατάσταση σε αυτή τη θέση χρησιμεύει ως αναπαράσταση επιπέδου πρότασης για την ταξινόμηση — περισσότερα παρακάτω.
- **[SEP] token (id 102)**: σηματοδοτεί το τέλος της πρότασης.
- **Padding και attention mask**: οι ακολουθίες συμπληρώνονται σε σταθερό μήκος (`max_length=128`). Το `attention_mask` σηματοδοτεί τα πραγματικά tokens με 1 και τις θέσεις padding με 0, ώστε το μοντέλο να αγνοεί τις θέσεις padding.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["sentence"],
)
print(tokenized_datasets)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1043
    })
    test: Dataset({
        features: ['label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1063
    })
})


### Τι καταφέραμε;

Κοιτάμε πάλι το παράδειγμα πριν περάσει το tokenization, και πώς το μετατρέπει το tokenization στην απαιτούμενη μορφή.

In [ ]:
# Inspect one example to confirm the structure
print(dataset["train"][0])
example = tokenized_datasets["train"][0]
print(example)
print("Keys:", list(example.keys()))
print("Label:", example["label"])
print("First 20 input_ids:   ", example["input_ids"][:20])
print("First 20 attention_mask:", example["attention_mask"][:20])
print("Sequence length:", len(example["input_ids"]))

{'sentence': "Our friends won't buy this analysis, let alone the next one we propose.", 'label': 1, 'idx': 0}
{'label': 1, 'idx': 0, 'input_ids': [101, 2256, 2814, 2180, 1005, 1056, 4965, 2023, 4106, 1010, 2292, 2894, 1996, 2279, 2028, 2057, 16599, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attenti

### Tokenizer

Ας το δούμε πιο προσεκτικά:

In [ ]:
print(tokenizer.encode("hello world"))
print(tokenizer.encode("hello, world"))
print(tokenizer.encode("hello, world."))

[101, 7592, 2088, 102]
[101, 7592, 1010, 2088, 102]
[101, 7592, 1010, 2088, 1012, 102]


In [ ]:
print(tokenizer.encode("time to move on."))

[101, 2051, 2000, 2693, 2006, 1012, 102]


Το πρώτο id είναι 101 ([CLS]) και το τελευταίο πραγματικό token πριν τα μηδενικά του padding είναι 102 ([SEP]). Το attention mask έχει τιμή 1 για όλα τα πραγματικά tokens και 0 για το padding.

## Φόρτωση του BERT

Φορτώνουμε το `bert-base-uncased` μέσω `AutoModelForMaskedLM` και εξάγουμε το attribute `.bert`. Αυτό μας δίνει τον κεντρικό κορμό που θέλουμε — τον encoder με 12 layers — χωρίς το masked language modeling head. Αυτά είναι τα pretrained βάρη που θα προσαρμόσουμε στο πρόβλημά μας.

In [ ]:
bert = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased").bert
print(bert)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
print(bert.config)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



## Το CLS Token ως Αναπαράσταση Πρότασης

Για να δούμε τι παράγει ο encoder, ας περάσουμε ένα tokenized παράδειγμα και ας εξετάσουμε το output.

In [ ]:
example = tokenized_datasets["train"][3]
input_ids    = torch.tensor(example["input_ids"]).unsqueeze(0)      # (1, 128)
attention_mask = torch.tensor(example["attention_mask"]).unsqueeze(0)  # (1, 128)

with torch.no_grad():
    output = bert(input_ids=input_ids, attention_mask=attention_mask)

print("last_hidden_state shape:", output.last_hidden_state.shape)
print("CLS vector (first 8 dims):", output.last_hidden_state[0, 0, :8])

last_hidden_state shape: torch.Size([1, 128, 768])
CLS vector (first 8 dims): tensor([ 0.2992,  0.1586, -0.1362, -0.1950, -0.8128, -0.2267,  0.6301,  0.6523])


Η έξοδος `last_hidden_state` έχει σχήμα `(batch_size, sequence_length, hidden_size)` — εδώ `(1, 128, 768)`. Κάθε μία από τις 128 θέσεις έχει μια 768-διάστατη συμφραζόμενη αναπαράσταση.

Η θέση ``[:,0,:]`` είναι το [CLS] token. Κατά την προεκπαίδευση του BERT, ένας αντικειμενικός στόχος πρόβλεψης της επόμενης πρότασης ώθησε το μοντέλο να συγκεντρώνει πληροφορίες επιπέδου πρότασης σε αυτή τη θέση. Ως αποτέλεσμα, το CLS διάνυσμα χρησιμοποιείται συνήθως ως αναπαράσταση πρότασης κατά το fine-tuning για ταξινόμηση: παρέχει μια μοναδική, σταθερού μεγέθους "περίληψη" ολόκληρης της ακολουθίας.

Αξίζει να σημειωθεί ότι αυτό δεν είναι κάτι μαγικό — το CLS διάνυσμα είναι χρήσιμο επειδή τα pretrained attention layers έχουν μάθει να δρομολογούν σχετικές πληροφορίες εκεί, αφού χρησιμοποιούμε encoder και όχι decoder (δείτε την [πρώτη διάλεξη στην ενότητα για μεγάλα γλωσσικά μοντέλα](https://youtu.be/K6QMaUGDPW0?list=PLXsmhnDvpjOQ3W4MzNgFobv4Gav3MTke_&index=18) για την σχετική συζήτηση).

Και φυσικά, η ποιότητα της αναπαράστασης εξαρτάται εξ ολοκλήρου από τον πλούτο της προεκπαίδευσης.

## Transfer Learning και νέος Ταξινομητής

Ορίζουμε το καινούργιο νευρωνικό δίκτυο που ουσιαστικά περιέχει τον κορμό του BERT, και έναν απλό ταξινομητή, ακριβώς όπως κάναμε και με την ResNet18 [σε αυτό το Colab Notebook](https://colab.research.google.com/drive/1tnIQoViTVbT-7XiJa791h492B_WuNw4t?usp=sharing) σε αυτήν την [διάλεξη της πρώτης σειράς](https://youtu.be/0NfdJbMd3eM):

1. Περνά την πρόταση από το pretrained BERT encoder
2. Εξάγει το τελικό CLS token: `last_hidden_state[:, 0, :]`
3. Το προβάλλει μέσω ενός γραμμικού layer (``nn.Linear(768,2)``) σε δύο logits (ένα ανά κλάση)

Όλα τα βάρη του BERT ενημερώνονται κατά το fine-tuning — πρόκειται για πλήρες fine-tuning, δεν "παγώνουμε" τα πρώτα στρώματα όπως κάναμε με το ResNet18.

Ο γραμμικός ταξινομητής αρχικοποιείται τυχαία και εκπαιδεύεται από την αρχή.

In [ ]:
class BertClassifier(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModelForMaskedLM.from_pretrained(model_name).bert
        self.classifier = torch.nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, **inputs):
        outputs = self.bert(**inputs) # batch_size X 128 X 768
        cls_output = outputs.last_hidden_state[:, 0, :]   # (batch, 768)
        logits = self.classifier(cls_output)              # (batch, 2)
        outputs["logits"] = logits
        return outputs

## Ρύθμιση Εκπαίδευσης

Χρησιμοποιούμε μικρό learning rate (2e-5), τυπικό για fine-tuning BERT — μεγάλες ενημερώσεις των βαρών θα κατέστρεφαν τις pretrained αναπαραστάσεις. Το cosine annealing schedule μειώνει ομαλά το learning rate κατά τη διάρκεια της εκπαίδευσης.

In [ ]:
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
EPOCHS        = 3
MAX_LENGTH    = 128

### Ετοιμάζουμε τα Δεδομένα: Dataloader & Collate

Πρέπει να ετοιμάσουμε τα δεδομένα για να μπορεί το PyTorch να τα χρησιμοποιήσει για την εκπαίδευση. Εδώ είναι που συνήθως ([όπως και στην πρώτη διάλεξη που το συζητήσαμε](https://youtu.be/c4Fyij73nBI)) χρησιμοποιούμε τα Data Loader, τα οποία ετοιμάζουν batches από δείγματα για την εκπαίδευση.

Για την εκπαίδευση των γλωσσικών μοντέλων το `DataLoader` χρειάζεται μια **collate function** που συναρμολογεί μεμονωμένα παραδείγματα από το dataset σε batches. Κάθε παράδειγμα είναι ένα Python dict από λίστες· το `CollateFN` τα μετατρέπει σε padded tensors στη μορφή που αναμένει το μοντέλο. Χρησιμοποιούμε `pad_sequence` αν και όλες οι ακολουθίες έχουν ήδη συμπληρωθεί στο `MAX_LENGTH` από τον tokenizer.

In [ ]:
@dataclass
class CollateFN:
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels, attention_mask = tuple(
            [instance[key] for instance in instances]
            for key in ("input_ids", "label", "attention_mask")
        )
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(x) for x in input_ids],
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        attention_masks = torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(x) for x in attention_mask],
            batch_first=True,
            padding_value=0,
        )
        return dict(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=torch.tensor(labels),
        )

In [ ]:
collate_fn = CollateFN(tokenizer=tokenizer)

train_dataloader = DataLoader(
    tokenized_datasets["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model     = BertClassifier("google-bert/bert-base-uncased").to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_dataloader))

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Εκπαίδευση

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            total_loss += criterion(outputs["logits"], batch["labels"]).item()
            correct += (outputs["logits"].argmax(dim=1) == batch["labels"]).sum().item()
            total += batch["labels"].size(0)
    return total_loss / len(dataloader), correct / total

# Evaluate before training
val_loss, val_acc = evaluate(model, val_dataloader, device)
print(f"Initial validation loss={val_loss:.4f}  Initial validation accuracy={val_acc:.4f}")



Initial validation loss=0.9199  Initial validation accuracy=0.3087


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    with tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS}") as pbar:
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            loss = criterion(outputs["logits"], batch["labels"])
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    val_loss, val_acc = evaluate(model, val_dataloader, device)
    print(f"Epoch {epoch + 1}/{EPOCHS}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

Epoch 1/3:   0%|          | 0/268 [00:00<?, ?it/s]

Epoch 1/3  val_loss=0.3954  val_acc=0.8284


Epoch 2/3:   0%|          | 0/268 [00:00<?, ?it/s]

Epoch 2/3  val_loss=0.4262  val_acc=0.8245


Epoch 3/3:   0%|          | 0/268 [00:00<?, ?it/s]

Epoch 3/3  val_loss=0.5309  val_acc=0.8207


## Συμπεράσματα

Μέσα από αυτή τη διαδικασία είδαμε στην πράξη πώς λειτουργεί το **Transfer Learning** με μεγάλα γλωσσικά μοντέλα όπως το BERT:

1. **Επαναχρησιμοποίηση Αναπαραστάσεων:** Αντί να εκπαιδεύσουμε ένα μοντέλο από το μηδέν (κάτι που θα απαιτούσε τεράστιο όγκο δεδομένων και υπολογιστικών πόρων), αξιοποιήσαμε τη γλωσσική «κατανόηση» που έχει ήδη το BERT.
2. **Ο ρόλος του [CLS] token:** Είδαμε πώς η αρχιτεκτονική συμπυκνώνει την πληροφορία ολόκληρης της πρότασης στο πρώτο token, το οποίο μπορούμε να χρησιμοποιήσουμε ως είσοδο σε έναν απλό ταξινομητή (linear layer).
3. **Αποδοτικό Fine-Tuning:** Με μόνο 3 epochs και ένα μικρό learning rate, καταφέραμε να προσαρμόσουμε τα βάρη του μοντέλου ώστε να λύνει το συγκεκριμένο πρόβλημα ταξινόμησης (CoLA), επιτυγχάνοντας γρήγορα αισθητά καλύτερη απόδοση από την τυχαία πρόβλεψη

Αυτή η προσέγγιση αποτελεί τη βάση για τις περισσότερες σύγχρονες εφαρμογές Επεξεργασίας Φυσικής Γλώσσας (NLP).